# GPT-SoVITS Kaggle API for Shinsekai

这个 notebook 用 Kaggle GPU 启动 GPT-SoVITS `api_v2.py`，并通过 Cloudflare Quick Tunnel 暴露给本机 Shinsekai 使用。

准备工作：
- Kaggle Notebook 右侧开启 `Accelerator: GPU` 和 `Internet: On`。
- 把 Shinsekai 导出的 `.char` 角色包作为 Kaggle Dataset 添加到当前 notebook；参考包结构可见本机 `Downloads/七海千秋.char`、`Downloads/和纱.char`。
- 运行到 tunnel 单元后，把输出的公网 URL 填到 Shinsekai 的 TTS URL。
- Notebook 会解析 `.char` 内的 `character.yaml`，把 `models/*.ckpt`、`models/*.pth`、参考音频和 prompt 转成 Kaggle 容器内路径。

Notebook 关闭或重启后 tunnel URL 会变化，需要重新运行并更新 Shinsekai 配置。

In [ ]:
from pathlib import Path
import os
import re
import json
import zipfile
import subprocess
import shutil
import sys
import threading
import time

WORKDIR = Path('/kaggle/working')
TEMP_ROOT = Path('/kaggle/temp') if Path('/kaggle/temp').exists() else WORKDIR
DOWNLOAD_DIR = TEMP_ROOT / 'gpt_sovits_downloads'
GPT_DIR = WORKDIR / 'GPT-SoVITS'
REPO_URL = 'https://github.com/RVC-Boss/GPT-SoVITS.git'
GIT_REF = 'main'

PORT = 9880
LOCAL_API = f'http://127.0.0.1:{PORT}'

# Kaggle 安装参数：HF / HF-Mirror / ModelScope
# Auto 会按 HF -> ModelScope -> HF-Mirror 尝试；单源可填 HF / ModelScope / HF-Mirror。
INSTALL_SOURCE = 'Auto'
# aria2 低于该速度会中止当前源并尝试下一个源。可改成 '512K' / '2M' / '0'。
MIN_DOWNLOAD_SPEED = '1M'
ARIA2_CONNECTIONS = 16
# auto 会根据 nvidia-smi 选择 CU128 / CU126；也可以手动写 CU128、CU126 或 CPU。
INSTALL_DEVICE = 'auto'
# 速度优先：没有 Kaggle GPU 时直接停止，避免静默退到 CPU 导致推理极慢。
REQUIRE_CUDA = True
# 只做 TTS API 时通常不需要 UVR5，训练/伴奏分离再打开。
INSTALL_UVR5 = False

# 可选：把 Linux/Kaggle 可用的 GPT-SoVITS 源码树或压缩包作为 Kaggle Dataset 添加。
# 注意：Shinsekai 的 GPT-SoVITS-v2pro-20250604.7z 是 Windows 整合包，不要直接作为 Kaggle 预构建包使用。
# 二者都留空时才在线安装 Linux 运行所需文件。
PREBUILT_GPT_SOVITS_DIR = ''      # 示例：'/kaggle/input/gpt-sovits-linux/GPT-SoVITS'
PREBUILT_GPT_SOVITS_ARCHIVE = ''  # 示例：'/kaggle/input/gpt-sovits-linux/GPT-SoVITS-linux.tar.gz'

# 可选：把官方预训练底座资源单独做成 Kaggle Dataset，避免每次从外网下载大文件。
# 支持目录中包含 pretrained_models、GPT_SoVITS/pretrained_models、G2PWModel、nltk_data、open_jtalk_dic_utf_8-1.11。
# Windows 整合包不能作为 PREBUILT_GPT_SOVITS_* 在 Kaggle 运行，但可以把其中 GPT_SoVITS/pretrained_models 作为 PRETRAINED_ASSETS_DIR 使用。
PRETRAINED_ASSETS_DIR = ''        # 示例：'/kaggle/input/gpt-sovits-pretrained'
# minimal-auto 先只下载公共底座；启动 API 前会按 .char 的 SoVITS 头版本补齐 v1/v2/v2Pro/v2ProPlus/v3/v4 专用文件。full-zip 下载官方完整 pretrained_models.zip。
PRETRAINED_DOWNLOAD_MODE = 'minimal-auto'
# 中文多音字增强模型较大；日文/英文角色通常不需要。中文角色需要时改成 True。
INSTALL_G2PW_MODEL = False

# Kaggle 免费输出空间约 19.5GB。下面这些开关会严格控制 /kaggle/working。
INSTALL_MISSING_SYSTEM_PACKAGES = False
UPGRADE_PIP_TOOLS = False
INSTALL_FASTER_WHISPER_EXTRA = False
NO_SOURCE_BUILDS = True
KEEP_DOWNLOAD_ARCHIVES = False
CLEAN_PIP_CACHE = True
PRUNE_GIT_METADATA = True
SPACE_WARN_GB = 4.0
API_LOG_TAIL_LINES = 200

# 推荐：把 Shinsekai 的 .char 角色包作为 Kaggle Dataset 添加进来。
# 留空会自动在 /kaggle/input 和 /kaggle/working 下找第一个 .char。
CHAR_FILE_PATH = ''       # 示例：'/kaggle/input/nanami-char/七海千秋.char'
CHARACTER_INDEX = 0       # 一个 .char 内有多个角色时选择第几个。
CHAR_EXTRACT_ROOT = WORKDIR / 'shinsekai_characters'

# 回退模式：没有 .char 时，才手动填这些 Kaggle 容器内路径。
MODEL_ROOT = Path('/kaggle/input')
GPT_MODEL_PATH = ''       # 示例：'/kaggle/input/my-gpt-sovits/model.ckpt'
SOVITS_MODEL_PATH = ''    # 示例：'/kaggle/input/my-gpt-sovits/model.pth'
REF_AUDIO_PATH = ''       # 示例：'/kaggle/input/my-gpt-sovits/ref.wav'

# 参考音频对应文本和语言。语言常用：zh / ja / en / yue / ko。
PROMPT_TEXT = ''
PROMPT_LANG = 'ja'
TEXT_LANG = 'ja'
TEST_TEXT = 'これは新世界の音声テストです。'

# TTS 请求参数：速度优先默认开启并行/分桶；长文本 batch_size=4 通常比 1 更快，OOM 时降回 1。
# speed_factor != 1.0 会让部分 GPT-SoVITS 版本关闭分桶/并行，速度优先时保持 1.0。
SHINSEKAI_COMPATIBLE_TTS_PAYLOAD = True
TEXT_SPLIT_METHOD = 'cut5'
TTS_BATCH_SIZE = 4
TTS_SPEED_FACTOR = 1.0
APPLY_CHAR_SPEECH_SPEED = False
TTS_EXTRA_PARAMS = {
    'parallel_infer': True,
    'split_bucket': True,
    'batch_threshold': 0.75,
    'super_sampling': False,
}
RUN_TTS_WARMUP = True
WARMUP_TEXT = 'テスト。'

# 参考音频清理：不会改 .char 原文件，只在 /kaggle/working 生成一个很小的 clean wav。
# 对游戏语音/压缩音频的底噪和齿音通常有帮助；如果听起来发闷，可以改成 False 对比。
CLEAN_REFERENCE_AUDIO = True
REF_AUDIO_HIGHPASS_HZ = 80
REF_AUDIO_LOWPASS_HZ = 12000
REF_AUDIO_DENOISE_NF = -25
# 可选：把 .char 里的 speech/<角色>/ 语音片段作为 aux_ref_audio_paths 传给 GPT-SoVITS。
# 这些片段不需要 prompt_text，但如果源音频很吵会拖累质量；默认关闭，测试 timbre 稳定性时再打开。
USE_AUX_REF_AUDIO = False
MAX_AUX_REF_AUDIO = 3

# 服务模式会启动 Cloudflare tunnel 并保持 notebook 运行；自动调试时可关闭 tunnel 并在 smoke test 后停止 API。
START_CLOUDFLARE_TUNNEL = True
RUN_TTS_SMOKE_TEST = True
# Kaggle 容器内自测默认走 127.0.0.1；公网 URL 主要给本机 Shinsekai 使用。
TEST_TTS_VIA_PUBLIC_URL = False
STOP_AFTER_TTS_SMOKE_TEST = False

WORKDIR.mkdir(parents=True, exist_ok=True)
print('WORKDIR:', WORKDIR)
print('TEMP_ROOT:', TEMP_ROOT)
print('Python:', sys.executable)

In [ ]:
def run(cmd, cwd=None, check=True, env=None):
    print('$', ' '.join(map(str, cmd)))
    proc = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if proc.stdout:
        print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed with code {proc.returncode}: {cmd}')
    return proc


def run_live(cmd, cwd=None, check=True, env=None):
    print('$', ' '.join(map(str, cmd)))
    # Inherit stdout/stderr so Kaggle shows download speed and ETA in real time.
    proc = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
    )
    if check and proc.returncode != 0:
        raise RuntimeError(f'command failed with code {proc.returncode}: {cmd}')
    return proc


def bytes_to_gb(value):
    return value / (1024 ** 3)


def dir_size_gb(path):
    path = Path(path)
    if not path.exists():
        return 0.0
    total = 0
    for p in path.rglob('*'):
        try:
            if p.is_file() or p.is_symlink():
                total += p.stat().st_size
        except OSError:
            pass
    return bytes_to_gb(total)


def disk_free_gb(path=WORKDIR):
    return bytes_to_gb(shutil.disk_usage(path).free)


def print_disk_report(label):
    usage = shutil.disk_usage(WORKDIR)
    print(f'\n[Disk] {label}')
    print(f'  /kaggle/working free: {bytes_to_gb(usage.free):.2f}GB / total: {bytes_to_gb(usage.total):.2f}GB')
    for p in [GPT_DIR, DOWNLOAD_DIR, WORKDIR / 'pip_cache', CHAR_EXTRACT_ROOT]:
        if p.exists():
            print(f'  {p}: {dir_size_gb(p):.2f}GB')
    if bytes_to_gb(usage.free) < SPACE_WARN_GB:
        print(f'  WARNING: free space below {SPACE_WARN_GB:.1f}GB; clean archives/cache or use /kaggle/input assets.')


def cleanup_path(path):
    path = Path(path)
    if not path.exists():
        return
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    else:
        try:
            path.unlink()
        except FileNotFoundError:
            pass
    print('Removed:', path)


def detect_install_device():
    if INSTALL_DEVICE and INSTALL_DEVICE.lower() != 'auto':
        return INSTALL_DEVICE
    proc = run(['nvidia-smi'], check=False)
    if proc.returncode != 0:
        return 'CPU'
    match = re.search(r'CUDA Version:\s*([0-9]+(?:\.[0-9]+)?)', proc.stdout or '')
    if match and float(match.group(1)) >= 12.8:
        return 'CU128'
    return 'CU126'


print('Detected install device:', detect_install_device())

In [ ]:
run([sys.executable, '--version'], check=False)
run(['nvidia-smi'], check=False)
run(['df', '-h', '/kaggle/working'], check=False)
print_disk_report('initial')

## 安装 GPT-SoVITS（Kaggle 兼容）

官方 `install.sh` 依赖 conda，Kaggle Notebook 默认没有 conda，所以这里不用 `install.sh`。本单元会优先复用 Kaggle 镜像里已经预装的系统包、Python 包、Torch/CUDA 栈和 NLTK 数据；只有缺失或版本不满足 GPT-SoVITS 官方要求时才安装。

Kaggle 免费 `/kaggle/working` 通常只有约 19.5GB，可用空间很紧。这个 notebook 默认会：不运行 `apt-get`、不升级 pip、不安装 `extra-req.txt` 里的 `faster-whisper`、禁止 pip 源码构建、把下载压缩包优先放到 `/kaggle/temp`、解压成功后删除下载压缩包、禁用 pip cache、清理 `.git` 元数据，并在关键阶段打印磁盘占用。

想进一步减少下载：
- 默认 `PRETRAINED_DOWNLOAD_MODE = 'minimal-auto'`，安装阶段只下载公共 BERT/HuBERT/langdetect；启动 API 前会读取 `.char` 里的 SoVITS 模型头版本，再按整合包配置补齐 v1/v2/v2Pro/v2ProPlus/v3/v4 对应底座文件，不再拉 4.56GB 的 `pretrained_models.zip`；需要完整底座时再改成 `full-zip`。
- 若你把官方 `pretrained_models`、`G2PWModel`、`nltk_data`、`open_jtalk_dic_utf_8-1.11` 做成 Kaggle Dataset，优先填写 `PRETRAINED_ASSETS_DIR`；notebook 会尽量软链，减少 `/kaggle/working` 占用和下载时间。
- 只有 Linux/Kaggle 兼容的 GPT-SoVITS 源码树或压缩包才填写 `PREBUILT_GPT_SOVITS_DIR` / `PREBUILT_GPT_SOVITS_ARCHIVE`；不要直接使用 Shinsekai Windows 整合包 `GPT-SoVITS-v2pro-20250604.7z`。
- `.char`、模型、参考音频也尽量放在 `/kaggle/input`，notebook 只抽取必要的 `models/*` 和配置到 `/kaggle/working`。
- `INSTALL_MISSING_SYSTEM_PACKAGES = False` 时不会为了 `aria2c` 额外 apt；若 Kaggle 当前镜像没有 `aria2c`，会用 `wget` 断点续传。
- `NO_SOURCE_BUILDS = True` 会给 pip 加 `--only-binary=:all:`，避免 `opencc` 这类源码编译；若某个必须包确实没有 wheel，再手动改成 `False`。
- `pyopenjtalk` 在 Python 3.12 上没有官方 wheel，notebook 会改装提供 `import pyopenjtalk` 兼容模块的 `pyopenjtalk-plus` wheel。
- `jieba_fast` 只有源码包且只是 `jieba` 的 C/SWIG 加速版本；Kaggle 默认跳过它，并自动生成 `jieba_fast/` package 兼容 shim 转发到纯 Python `jieba` 和 `jieba.posseg`。
- `funasr` 是 ASR/训练链路依赖，会拉取 Kaggle Python 3.12 下没有 wheel 的 `oss2`；TTS API 默认跳过它。
- `distance==0.1.3` 是 `g2p_en` 的纯 Python 小依赖但没有 wheel；notebook 会单独安装它，不开启 C/C++ 源码编译。

下载慢时优先看这几个配置：
- `INSTALL_SOURCE = 'Auto'`：按 HF -> ModelScope -> HF-Mirror 尝试。
- `MIN_DOWNLOAD_SPEED = '1M'`：低于该速度会放弃当前源并换下一个源；如果所有源都会被误杀，可以改成 `512K` 或 `0`。
- 所有下载命令使用实时输出；如果看不到速度，请展开 Kaggle 当前 cell 的输出区域。


In [ ]:
import importlib.metadata as importlib_metadata
import shutil
import tarfile

try:
    from packaging.requirements import Requirement
except Exception as exc:
    raise RuntimeError('Kaggle should include packaging; install packaging first if this fails.') from exc


MARKER_REQUIREMENT_PATTERNS = [
    (
        r'onnxruntime;\s*platform_machine\s*==\s*["\']aarch64["\']\s*or\s*platform_machine\s*==\s*["\']arm64["\']',
        'onnxruntime; platform_machine == "aarch64" or platform_machine == "arm64"',
    ),
    (
        r'onnxruntime-gpu;\s*platform_machine\s*==\s*["\']x86_64["\']\s*or\s*platform_machine\s*==\s*["\']AMD64["\']',
        'onnxruntime-gpu; platform_machine == "x86_64" or platform_machine == "AMD64"',
    ),
    (
        r'python_mecab_ko;\s*sys_platform\s*!=\s*["\']win32["\']',
        'python_mecab_ko; sys_platform != "win32"',
    ),
]

SOURCE_ONLY_OPTIONAL_REQUIREMENTS = {
    'jieba-fast': 'optional C/SWIG speed-up; GPT-SoVITS can use pure Python jieba instead',
    'funasr': 'ASR/training dependency; pulls oss2, which has no Python 3.12 wheel under --only-binary',
}

PURE_PYTHON_SOURCE_REQUIREMENTS = {
    'distance==0.1.3': 'pure Python dependency of g2p_en; no wheel is published on PyPI',
}

SYSTEM_PACKAGE_COMMANDS = {
    'ffmpeg': 'ffmpeg',
    'cmake': 'cmake',
    'make': 'build-essential',
    'gcc': 'build-essential',
    'g++': 'build-essential',
    'unzip': 'unzip',
    '7z': 'p7zip-full',
    'wget': 'wget',
    'aria2c': 'aria2',
}


def source_names(source):
    source = str(source or 'Auto').strip()
    if source == 'Auto':
        return ['HF', 'ModelScope', 'HF-Mirror']
    if source in {'HF', 'ModelScope', 'HF-Mirror'}:
        return [source]
    raise ValueError("INSTALL_SOURCE must be 'Auto', 'HF', 'HF-Mirror', or 'ModelScope'")


def source_base_for(source):
    source = str(source or 'HF').strip()
    if source == 'HF':
        return 'https://huggingface.co/XXXXRT/GPT-SoVITS-Pretrained/resolve/main'
    if source == 'HF-Mirror':
        return 'https://hf-mirror.com/XXXXRT/GPT-SoVITS-Pretrained/resolve/main'
    if source == 'ModelScope':
        return 'https://www.modelscope.cn/models/XXXXRT/GPT-SoVITS-Pretrained/resolve/master'
    raise ValueError("INSTALL_SOURCE must be 'Auto', 'HF', 'HF-Mirror', or 'ModelScope'")


def source_urls_for(source):
    base = source_base_for(source)
    return {
        'pretrained': f'{base}/pretrained_models.zip',
        'g2pw': f'{base}/G2PWModel.zip',
        'uvr5': f'{base}/uvr5_weights.zip',
        'nltk': f'{base}/nltk_data.zip',
        'open_jtalk': f'{base}/open_jtalk_dic_utf_8-1.11.tar.gz',
    }


def source_url_candidates(source):
    out = {}
    for source_name in source_names(source):
        urls = source_urls_for(source_name)
        for key, url in urls.items():
            out.setdefault(key, []).append((source_name, url))
    return out


def source_file_candidates(source, relative_path):
    rel = str(relative_path).replace('\\', '/').lstrip('/')
    return [(source_name, f'{source_base_for(source_name)}/{rel}') for source_name in source_names(source)]


def torch_index_url(device):
    if device == 'CU128':
        return 'https://download.pytorch.org/whl/cu128'
    if device == 'CU126':
        return 'https://download.pytorch.org/whl/cu126'
    if device == 'CPU':
        return 'https://download.pytorch.org/whl/cpu'
    raise ValueError(f'Kaggle install supports CU128, CU126, or CPU; got {device!r}')


def is_under(path, root):
    try:
        Path(path).resolve().relative_to(Path(root).resolve())
        return True
    except Exception:
        return False


def archive_looks_complete(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size <= 0:
        return False
    name = path.name.lower()
    try:
        if name.endswith('.zip'):
            with zipfile.ZipFile(path) as zf:
                return zf.testzip() is None
        if name.endswith('.tar.gz') or name.endswith('.tgz'):
            with tarfile.open(path, 'r:gz') as tf:
                tf.getmembers()
            return True
        if name.endswith('.7z'):
            if not shutil.which('7z'):
                return True
            result = run_live(['7z', 't', path], check=False)
            return result.returncode == 0
    except Exception as exc:
        print('Cached archive is incomplete or invalid:', path, exc)
        return False
    return True


def speed_limit_bytes(value):
    s = str(value or '').strip().upper()
    if not s or s == '0':
        return 0
    multiplier = 1
    if s.endswith('K'):
        multiplier = 1024
        s = s[:-1]
    elif s.endswith('M'):
        multiplier = 1024 ** 2
        s = s[:-1]
    elif s.endswith('G'):
        multiplier = 1024 ** 3
        s = s[:-1]
    try:
        return int(float(s) * multiplier)
    except ValueError:
        return 0


def download_file(candidates, output_path):
    output_path = Path(output_path)
    part_path = output_path.with_name(output_path.name + '.part')
    if output_path.exists() and output_path.stat().st_size > 0:
        if archive_looks_complete(output_path):
            print('Already downloaded:', output_path)
            return output_path
        print('Move incomplete cached archive to partial download:', part_path)
        if part_path.exists():
            part_path.unlink()
        output_path.replace(part_path)
    if isinstance(candidates, str):
        candidates = [('custom', candidates)]
    elif isinstance(candidates, tuple):
        candidates = [candidates]
    last_error = None
    for source_name, url in candidates:
        print(f'Downloading {output_path.name} from {source_name}: {url}')
        if shutil.which('aria2c'):
            cmd = [
                'aria2c',
                '--continue=true',
                f'--max-connection-per-server={ARIA2_CONNECTIONS}',
                f'--split={ARIA2_CONNECTIONS}',
                '--min-split-size=1M',
                '--file-allocation=none',
                '--summary-interval=20',
                '--timeout=30',
                '--connect-timeout=20',
                '--max-tries=5',
                '--retry-wait=5',
                f'--lowest-speed-limit={MIN_DOWNLOAD_SPEED}',
                '--allow-overwrite=true',
                '--auto-file-renaming=false',
                '-d', part_path.parent,
                '-o', part_path.name,
                url,
            ]
        elif shutil.which('curl'):
            cmd = [
                'curl',
                '-fL',
                '--retry', '5',
                '--retry-delay', '5',
                '--connect-timeout', '20',
                '--continue-at', '-',
                '--output', part_path,
            ]
            speed_limit = speed_limit_bytes(MIN_DOWNLOAD_SPEED)
            if speed_limit > 0:
                cmd.extend(['--speed-limit', str(speed_limit), '--speed-time', '60'])
            cmd.append(url)
        else:
            cmd = ['wget', '-c', '--tries=10', '--wait=5', '--read-timeout=40', '-O', part_path, url]
        result = run_live(cmd, check=False)
        if result.returncode == 0 and part_path.exists() and archive_looks_complete(part_path):
            part_path.replace(output_path)
            return output_path
        last_error = f'{source_name} failed with code {result.returncode}'
        print('Download source failed, trying next if available:', last_error)
    raise RuntimeError(f'All download sources failed for {output_path.name}: {last_error}')


def download_regular_file(candidates, output_path):
    output_path = Path(output_path)
    part_path = output_path.with_name(output_path.name + '.part')
    if output_path.exists() and output_path.stat().st_size > 0:
        print('Already downloaded:', output_path)
        return output_path
    if isinstance(candidates, str):
        candidates = [('custom', candidates)]
    elif isinstance(candidates, tuple):
        candidates = [candidates]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for source_name, url in candidates:
        print(f'Downloading {output_path.name} from {source_name}: {url}')
        if shutil.which('aria2c'):
            cmd = [
                'aria2c',
                '--continue=true',
                f'--max-connection-per-server={ARIA2_CONNECTIONS}',
                f'--split={ARIA2_CONNECTIONS}',
                '--min-split-size=1M',
                '--file-allocation=none',
                '--summary-interval=20',
                '--timeout=30',
                '--connect-timeout=20',
                '--max-tries=5',
                '--retry-wait=5',
                f'--lowest-speed-limit={MIN_DOWNLOAD_SPEED}',
                '--allow-overwrite=true',
                '--auto-file-renaming=false',
                '-d', part_path.parent,
                '-o', part_path.name,
                url,
            ]
        elif shutil.which('curl'):
            cmd = [
                'curl',
                '-fL',
                '--retry', '5',
                '--retry-delay', '5',
                '--connect-timeout', '20',
                '--continue-at', '-',
                '--output', part_path,
            ]
            speed_limit = speed_limit_bytes(MIN_DOWNLOAD_SPEED)
            if speed_limit > 0:
                cmd.extend(['--speed-limit', str(speed_limit), '--speed-time', '60'])
            cmd.append(url)
        else:
            cmd = ['wget', '-c', '--tries=10', '--wait=5', '--read-timeout=40', '-O', part_path, url]
        result = run_live(cmd, check=False)
        if result.returncode == 0 and part_path.exists() and part_path.stat().st_size > 0:
            part_path.replace(output_path)
            return output_path
        last_error = f'{source_name} failed with code {result.returncode}'
        print('Download source failed, trying next if available:', last_error)
    raise RuntimeError(f'All download sources failed for {output_path.name}: {last_error}')


def unzip_to(zip_path, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_dir)
    if not KEEP_DOWNLOAD_ARCHIVES and is_under(zip_path, WORKDIR):
        cleanup_path(zip_path)
    print_disk_report(f'after extracting {Path(zip_path).name}')


def extract_archive_to(archive_path, target_dir):
    archive_path = Path(archive_path)
    target_dir = Path(target_dir)
    if not archive_path.exists():
        raise FileNotFoundError(f'Archive not found: {archive_path}')
    target_dir.mkdir(parents=True, exist_ok=True)
    name = archive_path.name.lower()
    if name.endswith('.zip'):
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(target_dir)
    elif name.endswith(('.tar.gz', '.tgz')):
        with tarfile.open(archive_path, 'r:gz') as tf:
            tf.extractall(target_dir)
    elif name.endswith('.7z'):
        if not shutil.which('7z'):
            raise RuntimeError('7z is not available. Kaggle usually has p7zip-full; set INSTALL_MISSING_SYSTEM_PACKAGES=True if this runtime lacks it.')
        run_live(['7z', 'x', '-y', f'-o{target_dir}', archive_path])
    else:
        raise ValueError(f'Unsupported archive type: {archive_path}')
    if not KEEP_DOWNLOAD_ARCHIVES and is_under(archive_path, WORKDIR):
        cleanup_path(archive_path)
    print_disk_report(f'after extracting {archive_path.name}')


WINDOWS_BUNDLE_NAME_HINTS = ('gpt-sovits-v2pro', 'windows', 'win64', 'win-x64')
WINDOWS_BUNDLE_SENTINELS = (
    'runtime/python.exe',
    'runtime/scripts/python.exe',
    'runtime/lib/site-packages',
    'python.exe',
    'python310.dll',
    'python311.dll',
    'python312.dll',
    'ffmpeg.exe',
    'ffprobe.exe',
)
WINDOWS_RUNTIME_SUFFIXES = ('.dll', '.pyd')


def _norm_member_name(value):
    return str(value).replace('\\', '/').strip().lower().lstrip('./')


def _looks_like_windows_member(name):
    name = _norm_member_name(name)
    if any(marker in name for marker in WINDOWS_BUNDLE_SENTINELS):
        return True
    parts = name.split('/')
    if 'runtime' in parts and name.endswith(WINDOWS_RUNTIME_SUFFIXES):
        return True
    return False


def archive_has_windows_runtime(path):
    path = Path(path)
    name = path.name.lower()
    try:
        if name.endswith('.zip'):
            with zipfile.ZipFile(path) as zf:
                return any(_looks_like_windows_member(n) for n in zf.namelist())
        if name.endswith(('.tar.gz', '.tgz')):
            with tarfile.open(path, 'r:gz') as tf:
                return any(_looks_like_windows_member(m.name) for m in tf.getmembers())
        if name.endswith('.7z'):
            if not shutil.which('7z'):
                raise RuntimeError('Refuse to use an uninspectable .7z archive as a Kaggle prebuilt GPT-SoVITS tree. Use a Linux .tar.gz/.zip Dataset instead.')
            proc = subprocess.run(
                ['7z', 'l', '-ba', str(path)],
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                check=False,
            )
            if proc.returncode != 0:
                raise RuntimeError(f'Failed to inspect archive before extraction: {path}')
            return any(_looks_like_windows_member(line) for line in proc.stdout.splitlines())
    except RuntimeError:
        raise
    except Exception as exc:
        raise RuntimeError(f'Failed to inspect prebuilt archive before extraction: {path}: {exc}') from exc
    return False


def windows_bundle_reasons(path):
    path = Path(path)
    reasons = []
    lower_name = path.name.lower()
    if any(hint in lower_name for hint in WINDOWS_BUNDLE_NAME_HINTS):
        reasons.append(f'filename contains Windows bundle hint: {path.name}')
    if path.is_file() and archive_has_windows_runtime(path):
        reasons.append('archive contains Windows runtime files')
    if path.is_dir():
        for rel in WINDOWS_BUNDLE_SENTINELS:
            if (path / rel).exists():
                reasons.append(f'contains {rel}')
        runtime_dir = path / 'runtime'
        if runtime_dir.exists():
            for item in runtime_dir.rglob('*'):
                if item.is_file() and item.suffix.lower() in WINDOWS_RUNTIME_SUFFIXES:
                    reasons.append(f'contains Windows runtime binary: {item.relative_to(path)}')
                    break
    return reasons


def assert_not_windows_bundle(path):
    path = Path(path)
    if not path.exists():
        return
    reasons = windows_bundle_reasons(path)
    if reasons:
        details = '; '.join(reasons[:3])
        raise RuntimeError(
            'This looks like a Windows GPT-SoVITS integration package, not a Kaggle/Linux prebuilt tree. '
            f'Details: {details}. Use git clone, or provide a Linux/Kaggle Dataset with the source tree/pretrained assets.'
        )


def looks_like_gpt_sovits_root(path):
    path = Path(path)
    return (path / 'api_v2.py').is_file() and (path / 'GPT_SoVITS').is_dir()


def find_gpt_sovits_root(root):
    root = Path(root)
    assert_not_windows_bundle(root)
    if looks_like_gpt_sovits_root(root):
        return root
    if not root.exists():
        return None
    for api_path in root.rglob('api_v2.py'):
        parent = api_path.parent
        assert_not_windows_bundle(parent)
        if looks_like_gpt_sovits_root(parent):
            return parent
    return None


def install_system_packages():
    missing_commands = [cmd for cmd in SYSTEM_PACKAGE_COMMANDS if shutil.which(cmd) is None]
    if not missing_commands:
        print('System commands already available; skip apt-get.')
        return
    packages = sorted({SYSTEM_PACKAGE_COMMANDS[cmd] for cmd in missing_commands})
    print('Missing system commands:', ', '.join(missing_commands))
    if not INSTALL_MISSING_SYSTEM_PACKAGES:
        print('Skip apt-get by default. Kaggle Docker normally includes build-essential, unzip, cmake and p7zip-full.')
        return
    run_live(['apt-get', 'update'])
    run_live(['apt-get', 'install', '-y'] + packages)


def module_available(module_name):
    try:
        __import__(module_name)
        return True
    except Exception:
        return False


def requirement_project_name(line):
    match = re.match(r'([A-Za-z0-9][A-Za-z0-9._-]*)', line.strip())
    if not match:
        return ''
    return match.group(1).lower().replace('_', '-')


def parse_gpt_sovits_requirements(raw_path):
    text = Path(raw_path).read_text(encoding='utf-8')
    placeholders = {}
    for i, (pattern, requirement) in enumerate(MARKER_REQUIREMENT_PATTERNS):
        key = f'__MARKER_REQUIREMENT_{i}__'
        text = re.sub(pattern, f' {key} ', text)
        placeholders[key] = requirement
    text = re.sub(r'(^|\s)--no-binary[= ]opencc(?=\s|$)', ' ', text)
    requirements = []
    for token in text.split():
        token = placeholders.get(token, token).strip()
        if not token or token.startswith('-') or token.startswith('#'):
            continue
        if token == 'opencc':
            token = 'opencc-python-reimplemented'
        if token.startswith('pyopenjtalk'):
            token = token.replace('pyopenjtalk', 'pyopenjtalk-plus', 1)
        project_name = requirement_project_name(token)
        if project_name in SOURCE_ONLY_OPTIONAL_REQUIREMENTS and NO_SOURCE_BUILDS:
            print(f'Skip {token}: {SOURCE_ONLY_OPTIONAL_REQUIREMENTS[project_name]}')
            continue
        requirements.append(token)
    return requirements


def requirement_satisfied(line):
    req = Requirement(line)
    if req.marker is not None and not req.marker.evaluate():
        return True, 'environment marker does not apply'
    if req.url:
        return False, 'direct URL requirement'
    if req.extras:
        return False, 'extras may need additional packages'
    try:
        dist = importlib_metadata.distribution(req.name)
    except importlib_metadata.PackageNotFoundError:
        return False, 'not installed'
    if req.specifier and not req.specifier.contains(dist.version, prereleases=True):
        return False, f'installed {dist.version} does not satisfy {req.specifier}'
    return True, f'installed {dist.version}'


def prepare_kaggle_requirements():
    raw_path = GPT_DIR / 'requirements.txt'
    install_lines = []
    skipped = []
    for line in parse_gpt_sovits_requirements(raw_path):
        try:
            satisfied, reason = requirement_satisfied(line)
        except Exception as exc:
            satisfied, reason = False, f'could not check requirement: {exc}'
        if satisfied:
            skipped.append((line, reason))
        else:
            install_lines.append(line)
    out_path = WORKDIR / 'gpt_sovits_requirements_kaggle.txt'
    out_path.write_text('\n'.join(install_lines) + ('\n' if install_lines else ''), encoding='utf-8')
    print(f'Kaggle requirements written to: {out_path}')
    print(f'Requirements already satisfied/skipped: {len(skipped)}')
    print(f'Requirements to install: {len(install_lines)}')
    if install_lines:
        print('\n'.join('  - ' + line for line in install_lines))
    return out_path, install_lines


def pip_install(args, cwd=None, check=True):
    env = os.environ.copy()
    if CLEAN_PIP_CACHE:
        env['PIP_NO_CACHE_DIR'] = '1'
    else:
        env.setdefault('PIP_CACHE_DIR', str(WORKDIR / 'pip_cache'))
    env.setdefault('PIP_DISABLE_PIP_VERSION_CHECK', '1')
    return run_live([sys.executable, '-m', 'pip'] + list(map(str, args)), cwd=cwd, env=env, check=check)


def pip_install_args(*packages):
    args = ['install', '--prefer-binary']
    if NO_SOURCE_BUILDS:
        args.append('--only-binary=:all:')
    args.extend(packages)
    return args


def install_pure_python_source_requirements():
    if not NO_SOURCE_BUILDS:
        return
    missing = []
    for line, reason in PURE_PYTHON_SOURCE_REQUIREMENTS.items():
        try:
            satisfied, detail = requirement_satisfied(line)
        except Exception as exc:
            satisfied, detail = False, f'could not check requirement: {exc}'
        if satisfied:
            print(f'Whitelisted pure-Python source requirement already installed: {line} ({detail})')
        else:
            print(f'Install whitelisted pure-Python source requirement: {line} ({reason}; {detail})')
            missing.append(line)
    if missing:
        pip_install(['install', '--no-deps'] + missing)


def ensure_pkg_resources_python312_compatible():
    try:
        import pkg_resources
        print('pkg_resources OK:', getattr(pkg_resources, '__file__', '<unknown>'))
        return
    except AttributeError as exc:
        if 'ImpImporter' not in str(exc):
            raise
        print('Kaggle system pkg_resources is too old for Python 3.12; installing a newer setuptools wheel.')
    except Exception as exc:
        print('pkg_resources import failed; installing a newer setuptools wheel:', exc)
    sys.modules.pop('pkg_resources', None)
    pip_install(pip_install_args('setuptools>=69'))
    sys.modules.pop('pkg_resources', None)
    import pkg_resources
    print('pkg_resources fixed:', getattr(pkg_resources, '__file__', '<unknown>'))


def torch_ready(device):
    try:
        import torch
        import torchaudio
        print('Existing torch:', torch.__version__, 'cuda:', torch.version.cuda, 'available:', torch.cuda.is_available())
        print('Existing torchaudio:', torchaudio.__version__)
        return device == 'CPU' or torch.cuda.is_available()
    except Exception as exc:
        print('Torch stack check failed:', exc)
        return False


def ensure_jieba_fast_compat_shim():
    try:
        import jieba_fast
        import jieba_fast.posseg
        print('jieba_fast package already available:', getattr(jieba_fast, '__file__', '<unknown>'))
        return
    except Exception as exc:
        print('jieba_fast package check failed; creating pure Python compatibility package:', exc)
    if not module_available('jieba'):
        print('jieba is missing; installing pure Python jieba for jieba_fast compatibility shim.')
        pip_install(pip_install_args('jieba'))

    # Remove the older single-file shim, because `import jieba_fast.posseg` requires a package.
    old_file = GPT_DIR / 'jieba_fast.py'
    if old_file.exists():
        cleanup_path(old_file)
    old_cache = GPT_DIR / '__pycache__'
    if old_cache.exists():
        for cache_file in old_cache.glob('jieba_fast.*.pyc'):
            cleanup_path(cache_file)

    shim_dir = GPT_DIR / 'jieba_fast'
    shim_dir.mkdir(parents=True, exist_ok=True)
    (shim_dir / '__init__.py').write_text(
        """# Auto-generated Kaggle compatibility shim.\n"""
        """# GPT-SoVITS imports `jieba_fast`, but jieba_fast has no Kaggle Python 3.12 wheel.\n"""
        """import jieba as _jieba\n"""
        """from jieba import *  # noqa: F401,F403\n"""
        """for _name in dir(_jieba):\n"""
        """    if not _name.startswith('_'):\n"""
        """        globals()[_name] = getattr(_jieba, _name)\n"""
        """__all__ = [name for name in globals() if not name.startswith('_')]\n""",
        encoding='utf-8',
    )
    (shim_dir / 'posseg.py').write_text(
        """# Auto-generated Kaggle compatibility shim for jieba_fast.posseg.\n"""
        """import jieba.posseg as _posseg\n"""
        """from jieba.posseg import *  # noqa: F401,F403\n"""
        """for _name in dir(_posseg):\n"""
        """    if not _name.startswith('_'):\n"""
        """        globals()[_name] = getattr(_posseg, _name)\n"""
        """__all__ = [name for name in globals() if not name.startswith('_')]\n""",
        encoding='utf-8',
    )

    for module_name in list(sys.modules):
        if module_name == 'jieba_fast' or module_name.startswith('jieba_fast.'):
            sys.modules.pop(module_name, None)
    if str(GPT_DIR) not in sys.path:
        sys.path.insert(0, str(GPT_DIR))
    import jieba_fast
    import jieba_fast.posseg
    print('Created jieba_fast compatibility package:', shim_dir)
    print('jieba_fast shim module:', getattr(jieba_fast, '__file__', '<unknown>'))


def install_python_packages(device):
    if UPGRADE_PIP_TOOLS:
        pip_install(pip_install_args('-U', 'pip', 'setuptools', 'wheel'))
    else:
        print('Skip pip/setuptools/wheel upgrade; reuse Kaggle runtime tooling.')
    ensure_pkg_resources_python312_compatible()
    if torch_ready(device):
        print('Using Kaggle preinstalled torch stack; skip reinstalling large torch wheels.')
        if not module_available('torchcodec'):
            result = pip_install(pip_install_args('torchcodec', '--index-url', torch_index_url(device)), check=False)
            if result.returncode != 0:
                print('Warning: torchcodec install failed; continue because GPT-SoVITS API can run without it in many Kaggle runtimes.')
    else:
        pip_install(pip_install_args('torch', 'torchaudio', 'torchcodec', '--index-url', torch_index_url(device)))
    extra_req = GPT_DIR / 'extra-req.txt'
    if INSTALL_FASTER_WHISPER_EXTRA and extra_req.exists():
        pip_install(pip_install_args('-r', extra_req, '--no-deps'), cwd=GPT_DIR)
    else:
        print('Skip extra-req.txt by default; faster-whisper is not needed for TTS API.')
    install_pure_python_source_requirements()
    kaggle_requirements, install_lines = prepare_kaggle_requirements()
    if install_lines:
        pip_install(pip_install_args('-r', kaggle_requirements), cwd=GPT_DIR)
    else:
        print('All GPT-SoVITS requirements are already satisfied for this runtime.')
    ensure_jieba_fast_compat_shim()
    import opencc
    print('opencc module:', opencc.__file__)
    import torch
    print('Final torch:', torch.__version__, 'cuda available:', torch.cuda.is_available())
    if CLEAN_PIP_CACHE:
        run_live([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False)
        cleanup_path(WORKDIR / 'pip_cache')
    print_disk_report('after pip install')


def existing_nltk_data_dir():
    candidates = [Path('/usr/share/nltk_data'), Path(sys.prefix) / 'nltk_data']
    env_value = os.environ.get('NLTK_DATA', '')
    for part in env_value.split(os.pathsep):
        if part:
            candidates.insert(0, Path(part))
    for candidate in candidates:
        if candidate.exists() and any(candidate.iterdir()):
            return candidate
    return None


def pretrained_assets_root():
    if not PRETRAINED_ASSETS_DIR:
        return None
    root = Path(PRETRAINED_ASSETS_DIR)
    if not root.exists():
        raise FileNotFoundError(f'PRETRAINED_ASSETS_DIR does not exist: {root}')
    assert_not_windows_bundle(root)
    print('Using pretrained assets directory:', root)
    return root


def first_existing_asset(root, candidates):
    if root is None:
        return None
    for rel in candidates:
        path = root / rel
        if path.exists():
            return path
    return None


def link_or_copy_asset(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists():
        print('Asset already present:', dst)
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(src, dst, target_is_directory=src.is_dir())
        print('Linked asset:', dst, '->', src)
        return
    except Exception as exc:
        print('Symlink failed, copying asset instead:', exc)
    if src.is_dir():
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print('Copied asset:', dst)


COMMON_PRETRAINED_FILES = [
    'pretrained_models/chinese-hubert-base/config.json',
    'pretrained_models/chinese-hubert-base/preprocessor_config.json',
    'pretrained_models/chinese-hubert-base/pytorch_model.bin',
    'pretrained_models/chinese-roberta-wwm-ext-large/config.json',
    'pretrained_models/chinese-roberta-wwm-ext-large/pytorch_model.bin',
    'pretrained_models/chinese-roberta-wwm-ext-large/tokenizer.json',
    'pretrained_models/fast_langdetect/lid.176.bin',
]

# Mirrors GPT-SoVITS-v2pro-20250604/GPT_SoVITS/configs/tts_infer.yaml.
VERSION_PRETRAINED_FILES = {
    'v1': [
        'pretrained_models/s1bert25hz-2kh-longer-epoch=68e-step=50232.ckpt',
        'pretrained_models/s2G488k.pth',
    ],
    'v2': [
        'pretrained_models/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt',
        'pretrained_models/gsv-v2final-pretrained/s2G2333k.pth',
    ],
    'v2Pro': [
        'pretrained_models/s1v3.ckpt',
        'pretrained_models/v2Pro/s2Gv2Pro.pth',
        'pretrained_models/sv/pretrained_eres2netv2w24s4ep4.ckpt',
    ],
    'v2ProPlus': [
        'pretrained_models/s1v3.ckpt',
        'pretrained_models/v2Pro/s2Gv2ProPlus.pth',
        'pretrained_models/sv/pretrained_eres2netv2w24s4ep4.ckpt',
    ],
    'v3': [
        'pretrained_models/s1v3.ckpt',
        'pretrained_models/s2Gv3.pth',
        'pretrained_models/models--nvidia--bigvgan_v2_24khz_100band_256x/config.json',
        'pretrained_models/models--nvidia--bigvgan_v2_24khz_100band_256x/bigvgan_generator.pt',
    ],
    'v4': [
        'pretrained_models/s1v3.ckpt',
        'pretrained_models/gsv-v4-pretrained/s2Gv4.pth',
        'pretrained_models/gsv-v4-pretrained/vocoder.pth',
    ],
}

MINIMAL_V2_PRETRAINED_FILES = COMMON_PRETRAINED_FILES + VERSION_PRETRAINED_FILES['v2']


def pretrained_files_for_sovits(model_version, is_lora=False):
    model_version = str(model_version or '').strip()
    if model_version not in VERSION_PRETRAINED_FILES:
        return list(COMMON_PRETRAINED_FILES)
    files = list(COMMON_PRETRAINED_FILES) + list(VERSION_PRETRAINED_FILES[model_version])
    # v3/v4 LoRA checkpoints need the matching base generator; the table above already includes it.
    return list(dict.fromkeys(files))


def ensure_pretrained_file(rel, assets_root=None):
    dst = GPT_DIR / 'GPT_SoVITS' / rel
    if dst.exists() and dst.stat().st_size > 0:
        print('Pretrained asset present:', dst)
        return dst
    if assets_root is None:
        assets_root = pretrained_assets_root()
    asset = first_existing_asset(assets_root, [Path(rel), Path('GPT_SoVITS') / rel])
    if asset is not None:
        link_or_copy_asset(asset, dst)
    else:
        download_regular_file(source_file_candidates(INSTALL_SOURCE, rel), dst)
    return dst


def pretrained_complete(files):
    return all((GPT_DIR / 'GPT_SoVITS' / rel).exists() for rel in files)


def missing_pretrained(files):
    return [rel for rel in files if not (GPT_DIR / 'GPT_SoVITS' / rel).exists()]


def install_pretrained_files(files, assets_root, label):
    files = list(dict.fromkeys(files))
    for rel in files:
        ensure_pretrained_file(rel, assets_root=assets_root)
    missing = missing_pretrained(files)
    if missing:
        raise FileNotFoundError(f'{label} pretrained install incomplete: ' + ', '.join(missing))


def common_pretrained_complete():
    return pretrained_complete(COMMON_PRETRAINED_FILES)


def minimal_pretrained_complete():
    return pretrained_complete(MINIMAL_V2_PRETRAINED_FILES)


def install_common_pretrained_files(assets_root):
    install_pretrained_files(COMMON_PRETRAINED_FILES, assets_root, 'Common')


def install_minimal_v2_pretrained_files(assets_root):
    install_pretrained_files(MINIMAL_V2_PRETRAINED_FILES, assets_root, 'Minimal v2')


def install_pretrained_assets(urls):
    downloads = DOWNLOAD_DIR
    downloads.mkdir(parents=True, exist_ok=True)
    assets_root = pretrained_assets_root()

    pretrained_dir = GPT_DIR / 'GPT_SoVITS' / 'pretrained_models'
    full_pretrained_asset = first_existing_asset(assets_root, [
        Path('pretrained_models'),
        Path('GPT_SoVITS/pretrained_models'),
    ])
    if full_pretrained_asset is not None and not pretrained_dir.exists():
        link_or_copy_asset(full_pretrained_asset, pretrained_dir)

    mode = str(PRETRAINED_DOWNLOAD_MODE or 'minimal-auto').strip().lower()
    if mode in {'minimal-auto', 'auto', 'minimal'}:
        if common_pretrained_complete():
            print('Common pretrained files already present; version-specific files will be checked after reading the .char SoVITS header.')
        else:
            install_common_pretrained_files(assets_root)
    elif mode == 'minimal-v2':
        if minimal_pretrained_complete():
            print('Minimal v2 pretrained files already present.')
        else:
            install_minimal_v2_pretrained_files(assets_root)
    elif mode == 'full-zip':
        pretrained_marker = pretrained_dir / 'sv'
        if not pretrained_marker.exists():
            archive = download_file(urls['pretrained'], downloads / 'pretrained_models.zip')
            unzip_to(archive, GPT_DIR / 'GPT_SoVITS')
        else:
            print('Full pretrained models already present.')
    else:
        raise ValueError("PRETRAINED_DOWNLOAD_MODE must be 'minimal-auto', 'minimal-v2', or 'full-zip'")

    g2pw_dir = GPT_DIR / 'GPT_SoVITS' / 'text' / 'G2PWModel'
    if INSTALL_G2PW_MODEL:
        if not g2pw_dir.exists():
            asset = first_existing_asset(assets_root, [
                Path('G2PWModel'),
                Path('GPT_SoVITS/text/G2PWModel'),
            ])
            if asset is not None:
                link_or_copy_asset(asset, g2pw_dir)
            else:
                archive = download_file(urls['g2pw'], downloads / 'G2PWModel.zip')
                unzip_to(archive, GPT_DIR / 'GPT_SoVITS' / 'text')
        else:
            print('G2PWModel already present.')
    else:
        print('Skip G2PWModel; set INSTALL_G2PW_MODEL=True for Chinese polyphone support.')

    if INSTALL_UVR5:
        uvr5_dir = GPT_DIR / 'tools' / 'uvr5' / 'uvr5_weights'
        has_uvr5 = uvr5_dir.exists() and any(p.name != '.gitignore' for p in uvr5_dir.iterdir())
        if not has_uvr5:
            asset = first_existing_asset(assets_root, [
                Path('uvr5_weights'),
                Path('tools/uvr5/uvr5_weights'),
            ])
            if asset is not None:
                link_or_copy_asset(asset, uvr5_dir)
            else:
                archive = download_file(urls['uvr5'], downloads / 'uvr5_weights.zip')
                unzip_to(archive, GPT_DIR / 'tools' / 'uvr5')
        else:
            print('UVR5 weights already present.')

    nltk_dir = existing_nltk_data_dir()
    if nltk_dir is None:
        asset = first_existing_asset(assets_root, [Path('nltk_data')])
        if asset is not None:
            os.environ['NLTK_DATA'] = str(asset) + (os.pathsep + os.environ['NLTK_DATA'] if os.environ.get('NLTK_DATA') else '')
            print('Using NLTK data from pretrained assets:', asset)
        else:
            archive = download_file(urls['nltk'], downloads / 'nltk_data.zip')
            unzip_to(archive, sys.prefix)
    else:
        print('NLTK data already present:', nltk_dir)

    import pyopenjtalk
    pyopenjtalk_dir = Path(pyopenjtalk.__file__).resolve().parent
    openjtalk_marker = pyopenjtalk_dir / 'open_jtalk_dic_utf_8-1.11'
    if not openjtalk_marker.exists():
        asset = first_existing_asset(assets_root, [Path('open_jtalk_dic_utf_8-1.11')])
        if asset is not None:
            link_or_copy_asset(asset, openjtalk_marker)
        else:
            archive = download_file(urls['open_jtalk'], downloads / 'open_jtalk_dic_utf_8-1.11.tar.gz')
            with tarfile.open(archive, 'r:gz') as tf:
                tf.extractall(pyopenjtalk_dir)
            if not KEEP_DOWNLOAD_ARCHIVES and is_under(archive, WORKDIR):
                cleanup_path(archive)
            print_disk_report('after extracting open_jtalk')
    else:
        print('Open JTalk dictionary already present.')

    if not KEEP_DOWNLOAD_ARCHIVES:
        cleanup_path(downloads)
    print_disk_report('after pretrained assets')


def prepare_gpt_sovits_tree():
    global GPT_DIR
    if PREBUILT_GPT_SOVITS_DIR:
        root = find_gpt_sovits_root(PREBUILT_GPT_SOVITS_DIR)
        if root is None:
            raise FileNotFoundError(f'No GPT-SoVITS api_v2.py found under PREBUILT_GPT_SOVITS_DIR={PREBUILT_GPT_SOVITS_DIR!r}')
        GPT_DIR = root
        print('Using prebuilt GPT-SoVITS directory:', GPT_DIR)
        return
    if PREBUILT_GPT_SOVITS_ARCHIVE:
        assert_not_windows_bundle(PREBUILT_GPT_SOVITS_ARCHIVE)
        extract_root = WORKDIR / 'gpt_sovits_prebuilt'
        root = find_gpt_sovits_root(extract_root)
        if root is None:
            extract_archive_to(PREBUILT_GPT_SOVITS_ARCHIVE, extract_root)
            root = find_gpt_sovits_root(extract_root)
        if root is None:
            raise FileNotFoundError(f'No GPT-SoVITS api_v2.py found after extracting {PREBUILT_GPT_SOVITS_ARCHIVE!r}')
        GPT_DIR = root
        print('Using extracted GPT-SoVITS tree:', GPT_DIR)
        return
    if not GPT_DIR.exists():
        run_live(['git', 'clone', '--progress', '--depth', '1', '--branch', GIT_REF, REPO_URL, GPT_DIR])
    else:
        print('GPT-SoVITS already exists:', GPT_DIR)
    if PRUNE_GIT_METADATA and is_under(GPT_DIR, WORKDIR):
        cleanup_path(GPT_DIR / '.git')


prepare_gpt_sovits_tree()
install_device = detect_install_device()
print('Using Kaggle install device:', install_device)
install_system_packages()
install_python_packages(install_device)
install_pretrained_assets(source_url_candidates(INSTALL_SOURCE))
print('GPT-SoVITS Kaggle install completed.')


## 从 Shinsekai `.char` 角色包选择模型和参考音频

Notebook 会优先读取 `.char` 内的 `character.yaml`。`.char` 是 zip 包，项目导出结构为：`character.yaml`、`manifest.json`、`models/`、`sprites/<sprite_prefix>/`、`speech/<sprite_prefix>/`。

如果自动选择不符合预期，回到第一个配置单元手动填写 `CHAR_FILE_PATH` 或 `CHARACTER_INDEX` 后重新运行本单元。没有 `.char` 时才使用手动的 `GPT_MODEL_PATH`、`SOVITS_MODEL_PATH`、`REF_AUDIO_PATH`。

In [ ]:
import yaml
import wave
import struct
import math


def pick_first(root, suffixes):
    if not root.exists():
        return None
    suffixes = tuple(s.lower() for s in suffixes)
    matches = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in suffixes]
    return sorted(matches, key=lambda p: str(p).lower())[0] if matches else None


def find_char_file():
    if CHAR_FILE_PATH:
        p = Path(CHAR_FILE_PATH)
        if not p.exists():
            raise FileNotFoundError(f'CHAR_FILE_PATH not found: {p}')
        return p
    candidates = []
    for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if root.exists():
            candidates.extend(root.rglob('*.char'))
    return sorted(candidates, key=lambda p: str(p).lower())[0] if candidates else None


def package_relpath(path, field_name):
    raw = str(path or '').replace('\\', '/').strip()
    while raw.startswith('./'):
        raw = raw[2:]
    if not raw:
        return None
    if raw.startswith('/') or re.match(r'^[A-Za-z]:', raw):
        raise ValueError(f'{field_name} must be a relative .char package path: {path!r}')
    parts = raw.split('/')
    if any(part in ('', '.', '..') for part in parts):
        raise ValueError(f'{field_name} contains an unsafe path component: {path!r}')
    return Path(*parts)


def safe_extract_char(char_path, extract_root):
    extract_root.mkdir(parents=True, exist_ok=True)
    target = extract_root / char_path.stem
    if target.exists() and (target / 'character.yaml').exists():
        return target
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(char_path) as zf:
        names = []
        for info in zf.infolist():
            member = str(info.filename or '').replace('\\', '/').rstrip('/')
            if not member:
                continue
            package_relpath(member, 'zip member')
            names.append(member)
        yaml_name = 'character.yaml'
        if yaml_name not in names:
            raise FileNotFoundError(f'{char_path} does not contain character.yaml')
        for member in [yaml_name, 'manifest.json']:
            if member in names:
                zf.extract(member, target)
        characters = yaml.safe_load((target / yaml_name).read_text(encoding='utf-8')) or []
        needed = set()
        for character in characters:
            for key in ('gpt_model_path', 'sovits_model_path', 'refer_audio_path'):
                rel = package_relpath(character.get(key), key)
                if rel is not None:
                    needed.add(rel.as_posix())
        for member in sorted(needed):
            if member in names:
                zf.extract(member, target)
            else:
                print('Warning: referenced asset missing in .char:', member)
    print_disk_report(f'after extracting minimal char package {char_path.name}')
    return target


def resolve_char_asset(extract_dir, value, field_name):
    rel = package_relpath(value, field_name)
    if rel is None:
        return ''
    resolved = extract_dir / rel
    return resolved.as_posix() if resolved.exists() else ''


SOVITS_HEADER_VERSION_MAP = {
    b'00': ('v1', 'v1', False),
    b'01': ('v2', 'v2', False),
    b'02': ('v2', 'v3', False),
    b'03': ('v2', 'v3', True),
    b'04': ('v2', 'v4', True),
    b'05': ('v2', 'v2Pro', False),
    b'06': ('v2', 'v2ProPlus', False),
}


def inspect_sovits_header(path):
    p = Path(path)
    if not p.exists():
        return None
    with open(p, 'rb') as f:
        head = f.read(2)
    if head in SOVITS_HEADER_VERSION_MAP:
        version, model_version, is_lora = SOVITS_HEADER_VERSION_MAP[head]
        print('SoVITS header version:', model_version, 'base_version:', version, 'is_lora:', is_lora, 'head:', head)
        return {'version': version, 'model_version': model_version, 'is_lora': is_lora, 'head': head.hex()}
    size = p.stat().st_size
    if head == b'PK':
        if size < 82978 * 1024:
            version = model_version = 'v1'
        elif size < 700 * 1024 * 1024:
            version = model_version = 'v2'
        else:
            version, model_version = 'v2', 'v3'
        print('SoVITS old zip checkpoint inferred version:', model_version, 'size:', size)
        return {'version': version, 'model_version': model_version, 'is_lora': False, 'head': 'PK'}
    print('SoVITS unknown header:', head, 'size:', size)
    return None


def extract_aux_reference_audio(char_path, extract_dir, character):
    if not USE_AUX_REF_AUDIO or not char_path:
        return []
    sprite_prefix = _safe = str(character.get('sprite_prefix') or '').replace('\\', '/').strip('/').strip()
    if not sprite_prefix:
        return []
    aux_dir = Path(extract_dir) / 'aux_ref_audio'
    aux_dir.mkdir(parents=True, exist_ok=True)
    selected = []
    with zipfile.ZipFile(char_path) as zf:
        members = []
        for info in zf.infolist():
            member = str(info.filename or '').replace('\\', '/').rstrip('/')
            low = member.lower()
            if not low.startswith(f'speech/{sprite_prefix.lower()}/'):
                continue
            if not low.endswith(('.wav', '.mp3', '.flac', '.ogg', '.aac')):
                continue
            if Path(member).name.lower() == 'ref.wav':
                continue
            members.append(member)
        members = sorted(members, key=lambda m: zf.getinfo(m).file_size, reverse=True)[:MAX_AUX_REF_AUDIO]
        for member in members:
            package_relpath(member, 'aux voice')
            dst = aux_dir / Path(member).name
            if not dst.exists() or dst.stat().st_size != zf.getinfo(member).file_size:
                with zf.open(member) as src_f, open(dst, 'wb') as dst_f:
                    shutil.copyfileobj(src_f, dst_f)
            selected.append(dst.as_posix())
    print('AUX_REF_AUDIO_PATHS =', selected or '(disabled or none)')
    return selected


CHAR_FILE = find_char_file()
CHARACTER_NAME = ''
AUX_REF_AUDIO_PATHS = []
SOVITS_HEADER_INFO = None

if CHAR_FILE:
    CHAR_EXTRACT_DIR = safe_extract_char(CHAR_FILE, CHAR_EXTRACT_ROOT)
    with open(CHAR_EXTRACT_DIR / 'character.yaml', 'r', encoding='utf-8') as f:
        characters = yaml.safe_load(f) or []
    if not characters:
        raise ValueError(f'No character entries in {CHAR_FILE}')
    if CHARACTER_INDEX < 0 or CHARACTER_INDEX >= len(characters):
        raise IndexError(f'CHARACTER_INDEX out of range: {CHARACTER_INDEX}; entries={len(characters)}')
    character = characters[CHARACTER_INDEX]
    CHARACTER_NAME = str(character.get('name') or CHAR_FILE.stem)
    GPT_MODEL_PATH = resolve_char_asset(CHAR_EXTRACT_DIR, character.get('gpt_model_path'), 'gpt_model_path')
    SOVITS_MODEL_PATH = resolve_char_asset(CHAR_EXTRACT_DIR, character.get('sovits_model_path'), 'sovits_model_path')
    REF_AUDIO_PATH = resolve_char_asset(CHAR_EXTRACT_DIR, character.get('refer_audio_path'), 'refer_audio_path')
    PROMPT_TEXT = str(character.get('prompt_text') or PROMPT_TEXT or '')
    PROMPT_LANG = str(character.get('prompt_lang') or PROMPT_LANG or 'ja')
    TEXT_LANG = TEXT_LANG or PROMPT_LANG
    raw_speech_speed = character.get('speech_speed')
    try:
        CHAR_SPEECH_SPEED = float(raw_speech_speed) if raw_speech_speed not in (None, '') else None
    except (TypeError, ValueError):
        CHAR_SPEECH_SPEED = None
    if APPLY_CHAR_SPEECH_SPEED and CHAR_SPEECH_SPEED:
        TTS_SPEED_FACTOR = CHAR_SPEECH_SPEED
    else:
        TTS_SPEED_FACTOR = float(TTS_SPEED_FACTOR or 1.0)
    AUX_REF_AUDIO_PATHS = extract_aux_reference_audio(CHAR_FILE, CHAR_EXTRACT_DIR, character)
    manifest_path = CHAR_EXTRACT_DIR / 'manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}
    print('CHAR_FILE        =', CHAR_FILE)
    print('CHAR_EXTRACT_DIR =', CHAR_EXTRACT_DIR)
    print('CHARACTER_NAME   =', CHARACTER_NAME)
    if manifest:
        print('manifest original_paths keys =', sorted((manifest.get('original_paths') or {}).keys()))
else:
    print('No .char file found; using manual paths / suffix auto-discovery fallback.')
    gpt_model = Path(GPT_MODEL_PATH) if GPT_MODEL_PATH else pick_first(MODEL_ROOT, ['.ckpt'])
    sovits_model = Path(SOVITS_MODEL_PATH) if SOVITS_MODEL_PATH else pick_first(MODEL_ROOT, ['.pth'])
    ref_audio = Path(REF_AUDIO_PATH) if REF_AUDIO_PATH else pick_first(MODEL_ROOT, ['.wav', '.mp3', '.ogg', '.flac'])
    GPT_MODEL_PATH = str(gpt_model) if gpt_model else ''
    SOVITS_MODEL_PATH = str(sovits_model) if sovits_model else ''
    REF_AUDIO_PATH = str(ref_audio) if ref_audio else ''

print('GPT_MODEL_PATH   =', GPT_MODEL_PATH or '(not set)')
print('SOVITS_MODEL_PATH=', SOVITS_MODEL_PATH or '(not set)')
print('REF_AUDIO_PATH   =', REF_AUDIO_PATH or '(not set)')
print('PROMPT_TEXT      =', PROMPT_TEXT or '(not set)')
print('PROMPT_LANG      =', PROMPT_LANG)
print('TEXT_LANG        =', TEXT_LANG)
print('APPLY_CHAR_SPEECH_SPEED =', APPLY_CHAR_SPEECH_SPEED)
print('TTS_SPEED_FACTOR =', TTS_SPEED_FACTOR)
print('AUX_REF_AUDIO_PATHS =', AUX_REF_AUDIO_PATHS or '(disabled or none)')

for label, path, suffix in [
    ('GPT model', GPT_MODEL_PATH, '.ckpt'),
    ('SoVITS model', SOVITS_MODEL_PATH, '.pth'),
]:
    if not path:
        raise FileNotFoundError(f'{label} not configured. Upload a .char file or set the path manually.')
    if not path.lower().endswith(suffix):
        raise ValueError(f'{label} should end with {suffix}: {path}')
    if not Path(path).exists():
        raise FileNotFoundError(f'{label} not found in Kaggle container: {path}')
if not REF_AUDIO_PATH:
    raise FileNotFoundError('Reference audio not configured. Upload a .char file or set REF_AUDIO_PATH manually.')
if not Path(REF_AUDIO_PATH).exists():
    raise FileNotFoundError(f'Reference audio not found in Kaggle container: {REF_AUDIO_PATH}')
SOVITS_HEADER_INFO = inspect_sovits_header(SOVITS_MODEL_PATH)


def inspect_wav_audio(path, label):
    p = Path(path)
    if not p.exists():
        print(f'{label}: missing {p}')
        return None
    try:
        with wave.open(str(p), 'rb') as w:
            channels = w.getnchannels()
            sample_rate = w.getframerate()
            frames = w.getnframes()
            sample_width = w.getsampwidth()
            raw = w.readframes(frames)
    except Exception as exc:
        print(f'{label}: wave inspection skipped for {p}: {exc}')
        return None
    if sample_width != 2 or not raw:
        print(f'{label}: channels={channels} sr={sample_rate} duration={frames / max(sample_rate, 1):.2f}s sample_width={sample_width}')
        return None
    sample_count = len(raw) // 2
    samples = struct.unpack('<' + 'h' * sample_count, raw)
    peak = max(abs(x) for x in samples) if samples else 0
    rms = math.sqrt(sum(x * x for x in samples) / max(sample_count, 1)) if samples else 0
    clipped = sum(1 for x in samples if abs(x) >= 32760)
    near_silence = sum(1 for x in samples if abs(x) < 256)
    peak_db = 20 * math.log10(peak / 32768) if peak else -120
    rms_db = 20 * math.log10(rms / 32768) if rms else -120
    duration = frames / max(sample_rate, 1)
    print(f'{label}: {p}')
    print(f'  channels={channels} sr={sample_rate} duration={duration:.2f}s width={sample_width * 8}bit')
    print(f'  peak={peak_db:.2f} dBFS rms={rms_db:.2f} dBFS clipped={clipped} silence={near_silence / max(sample_count, 1):.1%}')
    if peak_db > -1.0:
        print('  note: reference peak is very hot; cleanup will leave headroom before GPT-SoVITS reads it.')
    if duration < 3 or duration > 10:
        print('  note: GPT-SoVITS reference audio usually works best around 3-10 seconds.')
    return {'channels': channels, 'sample_rate': sample_rate, 'duration': duration, 'peak_db': peak_db, 'rms_db': rms_db}


def make_clean_reference_audio(path):
    if not CLEAN_REFERENCE_AUDIO:
        return str(path)
    if not shutil.which('ffmpeg'):
        print('ffmpeg not found; use original reference audio.')
        return str(path)
    src = Path(path)
    dst = WORKDIR / 'shinsekai_ref_clean.wav'
    audio_filter = ','.join([
        f'highpass=f={REF_AUDIO_HIGHPASS_HZ}',
        f'lowpass=f={REF_AUDIO_LOWPASS_HZ}',
        f'afftdn=nf={REF_AUDIO_DENOISE_NF}',
        'dynaudnorm=f=75:g=7:p=0.90',
        'volume=0.90',
    ])
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(src), '-ac', '1', '-ar', '32000', '-af', audio_filter,
        str(dst),
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0 or not dst.exists() or dst.stat().st_size == 0:
        print('Reference cleanup failed; use original reference audio.')
        print(result.stdout[-1000:])
        return str(path)
    return str(dst)


inspect_wav_audio(REF_AUDIO_PATH, 'Original reference audio')
REF_AUDIO_PATH_EFFECTIVE = make_clean_reference_audio(REF_AUDIO_PATH)
if REF_AUDIO_PATH_EFFECTIVE != REF_AUDIO_PATH:
    inspect_wav_audio(REF_AUDIO_PATH_EFFECTIVE, 'Clean reference audio')
print('REF_AUDIO_PATH_EFFECTIVE =', REF_AUDIO_PATH_EFFECTIVE)

## 启动 GPT-SoVITS API

本单元会先生成 `/kaggle/working/tts_infer_shinsekai.yaml`，把 `.char` 解析出的 GPT/SoVITS 权重写入 `custom` 配置，再后台启动 `api_v2.py -a 0.0.0.0 -p 9880 -c ...`。日志写入 `/kaggle/working/gpt_sovits_api.log`。


In [ ]:
import requests

def stop_process(name):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()


def wait_for_api(base_url, timeout=360):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(base_url.rstrip('/') + '/docs', timeout=5)
            if response.status_code < 500:
                return
        except Exception as exc:
            last_error = exc
        time.sleep(3)
    raise TimeoutError(f'GPT-SoVITS API did not become ready. Last error: {last_error}')


def tail_file(path, lines=API_LOG_TAIL_LINES):
    path = Path(path)
    if not path.exists():
        return ''
    try:
        text = path.read_text(encoding='utf-8', errors='replace')
    except Exception as exc:
        return f'<failed to read {path}: {exc}>'
    return '\n'.join(text.splitlines()[-lines:])


def diagnose_api_log(path):
    text = tail_file(path, lines=1000)
    low = text.lower()
    findings = []
    if 'no module named' in low or 'modulenotfounderror' in low:
        findings.append('缺 Python 模块：安装依赖 cell 没完整成功，或 Kaggle runtime 重启后没有重新运行安装。')
    if 'pkgutil' in low and 'impimporter' in low and 'pkg_resources' in low:
        findings.append('Kaggle 的系统 pkg_resources/setuptools 太旧，和 Python 3.12 不兼容；重新运行安装 cell，它会定向安装新版 setuptools wheel。')
    if 'no space left on device' in low or 'disk quota' in low:
        findings.append('磁盘空间不足：删除 /kaggle/working/gpt_sovits_downloads、pip_cache，或重启 runtime 后重新按顺序运行。')
    if 'cuda out of memory' in low or 'outofmemoryerror' in low:
        findings.append('GPU 显存不足：关闭其他 notebook，或换更小模型/降低并发。')
    if 'torch' in low and ('cuda' in low or 'cudnn' in low) and ('error' in low or 'runtimeerror' in low):
        findings.append('PyTorch/CUDA 栈异常：优先使用 Kaggle 预装 torch；不要混装不匹配 CUDA wheel。')
    if 'fall back to default t2s_weights_path' in low or 'fall back to default vits_weights_path' in low:
        findings.append('API 启动时没有成功使用 .char 权重，已回退到默认预训练权重；需要用 runtime tts_infer_shinsekai.yaml 启动。')
    if 's1v3.ckpt' in low and ('no such file' in low or 'not found' in low):
        findings.append('缺少默认 s1v3.ckpt 通常不是必须下载的问题；应在启动 API 前把 .char 的 GPT/SoVITS 权重写入 custom 配置。')
    if 'pretrained' in low and ('not found' in low or 'no such file' in low):
        findings.append('预训练资源缺失：安装 cell 的 minimal/full pretrained 解压没有完成，或当前模型额外依赖 SV/底模文件。')
    if 'address already in use' in low or 'errno 98' in low:
        findings.append('端口被占用：先运行 stop_process 或重启 runtime，再启动 API。')
    if 'tts_infer.yaml' in low and ('not found' in low or 'no such file' in low):
        findings.append('GPT-SoVITS 目录不完整：git clone 或整合包解压不完整。')
    if not findings:
        findings.append('未命中内置诊断规则，请查看下面日志尾部的第一条 Traceback/ERROR。')
    print('\n[GPT-SoVITS API log diagnosis]')
    for item in findings:
        print('-', item)
    print('\n[GPT-SoVITS API log tail]')
    print(text or '<empty log>')


def detect_sovits_model_info(path):
    if not path:
        return None
    try:
        if str(GPT_DIR / 'GPT_SoVITS') not in sys.path:
            sys.path.insert(0, str(GPT_DIR / 'GPT_SoVITS'))
        if str(GPT_DIR) not in sys.path:
            sys.path.insert(0, str(GPT_DIR))
        from process_ckpt import get_sovits_version_from_path_fast
        version, model_version, if_lora_v3 = get_sovits_version_from_path_fast(str(path))
        info = {'version': version, 'model_version': model_version or version, 'is_lora': bool(if_lora_v3)}
        print('SoVITS startup version:', info)
        return info
    except Exception as exc:
        print('SoVITS startup version detection skipped:', repr(exc))
        info = globals().get('SOVITS_HEADER_INFO')
        if info:
            print('Use SoVITS header info from .char parse:', info)
            return dict(info)
        try:
            return inspect_sovits_header(path)
        except Exception as fallback_exc:
            print('SoVITS header fallback skipped:', repr(fallback_exc))
            return None


def ensure_sovits_runtime_assets(path):
    info = detect_sovits_model_info(path) or {}
    model_version = info.get('model_version') or info.get('version')
    is_lora = bool(info.get('is_lora'))
    required = pretrained_files_for_sovits(model_version, is_lora=is_lora)
    print('Required pretrained assets for SoVITS version:', model_version or '(unknown)', 'is_lora:', is_lora)
    for rel in required:
        ensure_pretrained_file(rel)
    missing = [rel for rel in required if not (GPT_DIR / 'GPT_SoVITS' / rel).exists()]
    if missing:
        raise FileNotFoundError('Version-specific pretrained install incomplete: ' + ', '.join(missing))
    return info


def detect_sovits_model_version(path):
    info = ensure_sovits_runtime_assets(path)
    if not info:
        return None
    return info.get('model_version') or info.get('version')


def build_runtime_tts_config(config_path):
    if not GPT_MODEL_PATH or not Path(GPT_MODEL_PATH).exists():
        raise FileNotFoundError(f'GPT_MODEL_PATH is not available before API startup: {GPT_MODEL_PATH!r}')
    if not SOVITS_MODEL_PATH or not Path(SOVITS_MODEL_PATH).exists():
        raise FileNotFoundError(f'SOVITS_MODEL_PATH is not available before API startup: {SOVITS_MODEL_PATH!r}')

    import yaml

    source_config = GPT_DIR / 'GPT_SoVITS' / 'configs' / 'tts_infer.yaml'
    if source_config.exists():
        with open(source_config, 'r', encoding='utf-8') as f:
            configs = yaml.safe_load(f) or {}
    else:
        configs = {}

    detected_version = detect_sovits_model_version(SOVITS_MODEL_PATH)
    allowed_versions = {'v1', 'v2', 'v3', 'v4', 'v2Pro', 'v2ProPlus'}
    base_version = detected_version if detected_version in allowed_versions else None

    custom = dict(configs.get('custom') or {})
    if not base_version:
        base_version = custom.get('version') if custom.get('version') in allowed_versions else None
    if not base_version:
        for candidate in ('v2ProPlus', 'v2Pro', 'v2', 'v3', 'v4', 'v1'):
            if candidate in configs:
                base_version = candidate
                break
    if not base_version:
        base_version = 'v2'

    base = dict(configs.get(base_version) or {})
    base.update(custom)
    base['version'] = base_version
    base['t2s_weights_path'] = str(Path(GPT_MODEL_PATH))
    base['vits_weights_path'] = str(Path(SOVITS_MODEL_PATH))

    try:
        import torch
        cuda_available = torch.cuda.is_available()
        print('CUDA available:', cuda_available)
        if cuda_available:
            print('CUDA device:', torch.cuda.get_device_name(0))
            base['device'] = 'cuda'
            base['is_half'] = True
        else:
            if REQUIRE_CUDA:
                raise RuntimeError('REQUIRE_CUDA=True but torch.cuda.is_available() is False. Enable Kaggle GPU accelerator before running this notebook.')
            base['device'] = 'cpu'
            base['is_half'] = False
    except Exception:
        if REQUIRE_CUDA:
            raise
        base.setdefault('device', 'cuda')
        base.setdefault('is_half', True)

    if not base.get('bert_base_path'):
        base['bert_base_path'] = 'GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large'
    if not base.get('cnhuhbert_base_path'):
        base['cnhuhbert_base_path'] = 'GPT_SoVITS/pretrained_models/chinese-hubert-base'

    configs['custom'] = base
    with open(config_path, 'w', encoding='utf-8') as f:
        yaml.safe_dump(configs, f, allow_unicode=True, sort_keys=False)

    print('Runtime TTS config:', config_path)
    print('  version          :', base.get('version'))
    print('  t2s_weights_path :', base.get('t2s_weights_path'))
    print('  vits_weights_path:', base.get('vits_weights_path'))
    print('  device/is_half   :', base.get('device'), base.get('is_half'))
    return config_path


stop_process('gpt_sovits_proc')
LOG_PATH = WORKDIR / 'gpt_sovits_api.log'
log_file = open(LOG_PATH, 'w', encoding='utf-8')
TTS_CONFIG_PATH = build_runtime_tts_config(WORKDIR / 'tts_infer_shinsekai.yaml')
api_cmd = [sys.executable, 'api_v2.py', '-a', '0.0.0.0', '-p', str(PORT), '-c', str(TTS_CONFIG_PATH)]
print('$', ' '.join(map(str, api_cmd)))
gpt_sovits_proc = subprocess.Popen(
    api_cmd,
    cwd=str(GPT_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
try:
    wait_for_api(LOCAL_API)
except Exception:
    try:
        log_file.flush()
    except Exception:
        pass
    diagnose_api_log(LOG_PATH)
    print_disk_report('after API startup failure')
    raise
print('GPT-SoVITS API is ready:', LOCAL_API)
print('Log:', LOG_PATH)

def inspect_sovits_checkpoint(path):
    if not path:
        return
    p = Path(path)
    print('SoVITS checkpoint path:', p)
    print('SoVITS checkpoint exists:', p.exists(), 'size:', p.stat().st_size if p.exists() else 'missing')
    try:
        if str(GPT_DIR / 'GPT_SoVITS') not in sys.path:
            sys.path.insert(0, str(GPT_DIR / 'GPT_SoVITS'))
        if str(GPT_DIR) not in sys.path:
            sys.path.insert(0, str(GPT_DIR))
        from process_ckpt import get_sovits_version_from_path_fast, load_sovits_new
        version, model_version, if_lora_v3 = get_sovits_version_from_path_fast(str(p))
        print('SoVITS detected version:', version, 'model_version:', model_version, 'if_lora_v3:', if_lora_v3)
        ckpt = load_sovits_new(str(p))
        print('SoVITS checkpoint keys:', list(ckpt.keys()))
        cfg = ckpt.get('config')
        weight = ckpt.get('weight') or {}
        print('SoVITS config type:', type(cfg).__name__)
        if isinstance(cfg, dict):
            print('SoVITS config model keys:', sorted((cfg.get('model') or {}).keys()))
            print('SoVITS config data:', cfg.get('data'))
            print('SoVITS config train:', cfg.get('train'))
        for key in ['enc_p.text_embedding.weight', 'enc_p.emb.weight', 'dec.conv_pre.weight']:
            value = weight.get(key)
            print('SoVITS weight', key, None if value is None else tuple(value.shape))
    except Exception as exc:
        print('SoVITS checkpoint inspection failed:', repr(exc))


def set_weight(endpoint, path):
    if not path:
        return
    response = requests.get(f'{LOCAL_API}/{endpoint}', params={'weights_path': path}, timeout=180)
    print(endpoint, response.status_code)
    print(response.text[:4000])
    if response.status_code >= 400:
        try:
            payload = response.json()
        except Exception:
            payload = {}
        exception_text = payload.get('Exception') or payload.get('message') or response.text
        raise RuntimeError(f'{endpoint} failed for {path}: {exception_text}')
    response.raise_for_status()


try:
    inspect_sovits_checkpoint(SOVITS_MODEL_PATH)
    set_weight('set_gpt_weights', GPT_MODEL_PATH)
    set_weight('set_sovits_weights', SOVITS_MODEL_PATH)
except Exception:
    try:
        log_file.flush()
    except Exception:
        pass
    diagnose_api_log(LOG_PATH)
    print_disk_report('after model switch failure')
    raise

print('\nFill these paths into Shinsekai character voice settings:')
print('character_name   :', CHARACTER_NAME or '(manual paths)')
print('gpt_model_path   :', GPT_MODEL_PATH)
print('sovits_model_path:', SOVITS_MODEL_PATH)
print('refer_audio_path :', globals().get('REF_AUDIO_PATH_EFFECTIVE', REF_AUDIO_PATH))
print('prompt_text      :', PROMPT_TEXT)
print('prompt_lang      :', PROMPT_LANG)

## 暴露公网 URL

Cloudflare Quick Tunnel 不需要账号，但 URL 每次启动都会变化。保持这个 notebook 运行，Shinsekai 才能访问。

In [ ]:
PUBLIC_URL = LOCAL_API
if START_CLOUDFLARE_TUNNEL:
    CLOUDFLARED_URLS = [('GitHub', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')]
    CLOUDFLARED = WORKDIR / 'cloudflared'
    CLOUDFLARED_CACHE = Path('/tmp/cloudflared')

    def looks_like_elf(path):
        try:
            return Path(path).is_file() and Path(path).read_bytes()[:4] == b'\x7fELF'
        except Exception:
            return False

    def clear_cloudflared_files():
        for path in (CLOUDFLARED, CLOUDFLARED_CACHE):
            try:
                Path(path).unlink(missing_ok=True)
            except Exception as exc:
                print('Failed to delete cloudflared file:', path, repr(exc))

    def ensure_cloudflared(force_redownload=False):
        if force_redownload:
            clear_cloudflared_files()
        if not looks_like_elf(CLOUDFLARED_CACHE):
            if CLOUDFLARED_CACHE.exists():
                print('Existing cloudflared cache is not a Linux executable; re-downloading:', CLOUDFLARED_CACHE)
                CLOUDFLARED_CACHE.unlink(missing_ok=True)
            download_regular_file(CLOUDFLARED_URLS, CLOUDFLARED_CACHE)
        if not looks_like_elf(CLOUDFLARED_CACHE):
            raise RuntimeError(f'cloudflared download is not a Linux executable: {CLOUDFLARED_CACHE}')
        if not looks_like_elf(CLOUDFLARED):
            if CLOUDFLARED.exists():
                print('Existing cloudflared target is not a Linux executable; replacing:', CLOUDFLARED)
                CLOUDFLARED.unlink(missing_ok=True)
            shutil.copy2(CLOUDFLARED_CACHE, CLOUDFLARED)
        CLOUDFLARED.chmod(0o755)
        return CLOUDFLARED

    def start_cloudflared():
        ensure_cloudflared()
        cloudflared_cmd = [str(CLOUDFLARED), 'tunnel', '--url', LOCAL_API, '--no-autoupdate']
        print('$', ' '.join(cloudflared_cmd))
        try:
            return subprocess.Popen(
                cloudflared_cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
        except (PermissionError, OSError) as exc:
            print('cloudflared launch failed; deleting cached binary and re-downloading:', repr(exc))
            ensure_cloudflared(force_redownload=True)
            print('$', ' '.join(cloudflared_cmd))
            return subprocess.Popen(
                cloudflared_cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )

    stop_process('cloudflared_proc')
    cloudflared_proc = start_cloudflared()
    cloudflared_lines = []

    def drain_cloudflared():
        for line in cloudflared_proc.stdout:
            cloudflared_lines.append(line)
            print(line, end='')


    threading.Thread(target=drain_cloudflared, daemon=True).start()

    deadline = time.time() + 90
    pattern = re.compile(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com')
    while time.time() < deadline:
        joined = ''.join(cloudflared_lines)
        match = pattern.search(joined)
        if match:
            PUBLIC_URL = match.group(0).rstrip('/')
            break
        if cloudflared_proc.poll() is not None:
            raise RuntimeError('cloudflared exited before a public URL was created')
        time.sleep(1)

    if not PUBLIC_URL or PUBLIC_URL == LOCAL_API:
        raise TimeoutError('Could not find a trycloudflare.com URL in cloudflared output')

    print('\nPUBLIC_URL =', PUBLIC_URL)
    print('\nIn Shinsekai API settings:')
    print('TTS engine      : Kaggle GPT-SoVITS')
    print('GPT SoVITS URL  :', PUBLIC_URL)
    print('GPT SoVITS dir  : leave empty')
else:
    stop_process('cloudflared_proc')
    print('Skip Cloudflare tunnel; PUBLIC_URL uses local API for smoke test:', PUBLIC_URL)


## 连通性测试

如果已经设置 `REF_AUDIO_PATH` 和 `PROMPT_TEXT`，本单元会请求 `/tts` 并保存一段测试 wav。默认使用 `LOCAL_API`，避免 Kaggle 容器内解析 Cloudflare Quick Tunnel URL 失败；要测试公网 URL 时再把 `TEST_TTS_VIA_PUBLIC_URL=True`。

In [ ]:
from IPython.display import Audio, display

try:
    public_url = globals().get('PUBLIC_URL')
    base_url = public_url if TEST_TTS_VIA_PUBLIC_URL and public_url else LOCAL_API
    print('TTS smoke test URL:', base_url)
    if not RUN_TTS_SMOKE_TEST:
        print('Skip TTS smoke test: RUN_TTS_SMOKE_TEST=False.')
    elif not REF_AUDIO_PATH or not PROMPT_TEXT:
        print('Skip TTS test: set REF_AUDIO_PATH and PROMPT_TEXT first.')
    else:
        payload = {
            'text': TEST_TEXT,
            'text_lang': TEXT_LANG,
            'ref_audio_path': globals().get('REF_AUDIO_PATH_EFFECTIVE', REF_AUDIO_PATH),
            'prompt_text': PROMPT_TEXT,
            'prompt_lang': PROMPT_LANG,
            'text_split_method': TEXT_SPLIT_METHOD,
            'batch_size': TTS_BATCH_SIZE,
            'speed_factor': TTS_SPEED_FACTOR,
        }
        if USE_AUX_REF_AUDIO:
            payload['aux_ref_audio_paths'] = AUX_REF_AUDIO_PATHS
        if not SHINSEKAI_COMPATIBLE_TTS_PAYLOAD:
            payload.update({'media_type': 'wav', 'streaming_mode': False})
        payload.update(TTS_EXTRA_PARAMS)
        print('TTS payload:', json.dumps({k: v for k, v in payload.items() if k != 'prompt_text'}, ensure_ascii=False, indent=2))
        if RUN_TTS_WARMUP:
            warmup_payload = dict(payload)
            warmup_payload['text'] = WARMUP_TEXT
            warmup_started = time.time()
            warmup_response = requests.post(LOCAL_API.rstrip('/') + '/tts', json=warmup_payload, timeout=300)
            warmup_elapsed = time.time() - warmup_started
            print(f'warmup status: {warmup_response.status_code}, elapsed: {warmup_elapsed:.2f}s, bytes: {len(warmup_response.content)}')
            warmup_response.raise_for_status()
        try:
            started = time.time()
            response = requests.post(base_url.rstrip('/') + '/tts', json=payload, timeout=300)
            elapsed = time.time() - started
        except requests.RequestException as exc:
            if base_url != LOCAL_API:
                print('Public tunnel smoke test failed, retrying LOCAL_API:', repr(exc))
                base_url = LOCAL_API
                started = time.time()
                response = requests.post(base_url.rstrip('/') + '/tts', json=payload, timeout=300)
                elapsed = time.time() - started
            else:
                raise
        print(f'status: {response.status_code}, elapsed: {elapsed:.2f}s')
        print(response.text[:500] if response.status_code != 200 else 'audio bytes: ' + str(len(response.content)))
        response.raise_for_status()
        out = WORKDIR / 'shinsekai_gpt_sovits_test.wav'
        out.write_bytes(response.content)
        print('Saved:', out)
        try:
            display(Audio(str(out)))
        except Exception as exc:
            print('Audio preview skipped:', exc)
finally:
    if STOP_AFTER_TTS_SMOKE_TEST:
        print('STOP_AFTER_TTS_SMOKE_TEST=True; stopping background services.')
        stop_process('cloudflared_proc')
        stop_process('gpt_sovits_proc')


## Shinsekai 填写说明

API 设定：
- `TTS 引擎` 选择 `Kaggle GPT-SoVITS`。
- TTS URL 填上面输出的 `PUBLIC_URL`。
- `GPT SoVITS 目录` 会在 Kaggle 模式下禁用并保存为空；不要选择普通 `GPT SoVITS`，否则 Shinsekai 会按本地/通用 GPT-SoVITS 流程切换模型路径。

角色语音模型：
- 先在 Shinsekai 本地导入同一个 `.char` 角色包，保留立绘和角色配置。
- 默认 `Kaggle GPT-SoVITS` 不会向远端请求切换权重，而是使用 notebook 已加载的 `.char` 模型和 `/kaggle/working/shinsekai_ref_clean.wav` 参考音频。
- 只有需要多角色动态切权重时，才在 TTS 额外设置里打开 `switch_weights`，并填写 Kaggle 容器内的 `remote_gpt_model_path` / `remote_sovits_model_path` / `remote_ref_audio_path`。
- `Prompt 文本` 和 `Prompt 语言` 默认从 notebook 解析的 `.char` 中读取；如手动改动，必须和远端参考音频内容一致。

注意：远端 Kaggle 模式下，GPT-SoVITS 只能读取 Kaggle 容器里的文件。不要把本机路径填到 Kaggle 额外设置；如果角色立绘绑定了本机语音文本，建议清空该立绘的 `voice_text`，否则可能覆盖全局参考音频。